In [1]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import os

### AES CTR (Counter Mode)

In [2]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

def int_to_block(n: int) -> bytes:
    """Counter value → 16-byte big-endian block."""
    return n.to_bytes(16, "big")

def aes_ecb_encrypt_block(key: bytes, block: bytes) -> bytes:
    """Encrypt a single 16-byte block with AES-ECB (used as the core primitive)."""
    cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
    enc = cipher.encryptor()
    return enc.update(block) + enc.finalize()

def aes_ctr_manual(key: bytes, nonce: int, data: bytes) -> bytes:
    """AES-CTR encrypt/decrypt (same operation)."""
    output = b""
    for i, offset in enumerate(range(0, len(data), 16)):
        block = data[offset:offset + 16]
        counter_block = int_to_block(nonce + i)   # nonce + counter
        keystream = aes_ecb_encrypt_block(key, counter_block)
        output += xor_bytes(block, keystream)      # XOR with plaintext/ciphertext
    return output

def aes_ctr_decrypt_block(key: bytes, nonce: int, ciphertext: bytes, block_index: int) -> bytes:
    offset = block_index * 16
    block = ciphertext[offset: offset+16]

    counter_block = int_to_block(nonce+block_index)
    keystream = aes_ecb_encrypt_block(key, counter_block)

    return xor_bytes(block, keystream)

In [ ]:
# key   = os.urandom(32)   # 256-bit key
key = bytes.fromhex("46cd95d06285916eb2b9bf48733c7460")  # 16 bytes → AES-128
nonce = int.from_bytes(os.urandom(12), "big")

plaintext = b"Hello, AES-CTR! BLOCK1_CONTENT!!"

ciphertext = aes_ctr_manual(key, nonce, plaintext)
decrypted  = aes_ctr_manual(key, nonce, ciphertext)

block0 = aes_ctr_decrypt_block(key, nonce, ciphertext, block_index=0)

print("Plaintext :", plaintext)
print("Ciphertext:", ciphertext.hex())
print("Decrypted :", decrypted)
print("Match     :", plaintext == decrypted)
print(f"Block 0  : {block0}")

Plaintext : b'Hello, AES-CTR! BLOCK1_CONTENT!!'
Ciphertext: 0956deae530a542f51fb0e308edd186fcb31ad3a49cb9247f0946a6a357917a8
Decrypted : b'Hello, AES-CTR! BLOCK1_CONTENT!!'
Match      : True
Block 0: b'Hello, AES-CTR! '
